# Annexe 2 : GVCF à grande échelle - la procédure en 5 étapes
*École de bioinformatique INRAe-IFB-INSERM 2026*

* Maria BERNARD - INRAE

Cette annexe complète la section **III.3.2 GATK avec sortie GVCF** du TP principal. Elle détaille la procédure en **5 étapes**, recommandée par le Broad Institute dès lors que l'on travaille sur un **grand nombre d'échantillons** et/ou un **grand génome**, en remplacement de la procédure en 3 étapes (`HaplotypeCaller` → `CombineGVCFs` → `GenotypeGVCFs`) vue en TP.

source :
* [GATK - GenomicsDBImport](https://gatk.broadinstitute.org/hc/en-us/articles/30332006200603-GenomicsDBImport)
* [GATK - The logic of joint calling for germline short variants](https://gatk.broadinstitute.org/hc/en-us/articles/360035890431-The-logic-of-joint-calling-for-germline-short-variants)


## I - Pourquoi remplacer `CombineGVCFs` par `GenomicsDBImport` ?

`CombineGVCFs` (procédure en 3 étapes du TP principal) fusionne l'ensemble des GVCF individuels en **un seul** fichier GVCF multi-échantillon, en relisant et réécrivant l'intégralité des données à chaque appel.

<div style="background-color: #fff3cd; border-left: 6px solid #ffc107; color: #856404; padding: 10px; border-radius: 4px;">
    ⚠️Warning<br>
    Cette approche ne passe pas à l'échelle : sur un grand génome ou avec beaucoup d'échantillons, le fichier GVCF combiné devient énorme, le temps de calcul explose, et ajouter un seul nouvel échantillon oblige à tout refusionner.
</div>

[GenomicsDBImport](https://gatk.broadinstitute.org/hc/en-us/articles/30332006200603-GenomicsDBImport) résout ce problème en important les GVCF individuels dans une **base de données génomique**, organisée par intervalle du génome plutôt que par fichier plat. Cela permet de :
* **paralléliser** le génotypage sur des sous-régions du génome (plusieurs jobs simultanés, un par intervalle) ;
* **ajouter incrémentalement** de nouveaux échantillons à une base existante, sans tout refaire.

<div style="background-color: #e7f3fe; border-left: 6px solid #2196F3; padding: 10px;">
    ℹ️ <strong>Remarque</strong><br>
    Rappel du TP principal : la recommandation du Broad Institute est de passer à cette procédure en 5 étapes dès que l'on dépasse quelques dizaines d'échantillons ou que l'on travaille sur un grand génome.
</div>


## II - Vue d'ensemble du pipeline

```
 sample1.g.vcf    sample2.g.vcf    ...    sampleN.g.vcf      (HaplotypeCaller -ERC GVCF, un par échantillon)
        │                 │                      │
        │                 │                      │
        │                 │                      │
        │ ┌────────────────────────────────────┐ │
        │ │ SplitIntervals (du génome entier)  │ │
        │ │  →  intervalle I1, I2, ... , Ik    │ │
        │ └────────────────────────────────────┘ │
        ▼               ▼ │                      ▼
        ┌─────────────────┼──────────────────────┐
        ▼                 ▼                      ▼
  GenomicsDBImport   GenomicsDBImport     GenomicsDBImport      (un par intervalle Ii, tous les échantillons)
     (I1)                (I2)                  (Ik)
        │                 │                     │
        ▼                 ▼                     ▼
  GenotypeGVCFs      GenotypeGVCFs         GenotypeGVCFs         (un par intervalle Ii)
     (I1)                (I2)                  (Ik)
        │                 │                     │
        └─────────────────┼─────────────────────┘
                          ▼
                     GatherVcfs
                          │
                          ▼
          cohort_genome.vcf.gz   (VCF multi-échantillon, génome entier)
```

Soit, dans l'ordre :
1. **HaplotypeCaller** en mode GVCF, un par échantillon (déjà vu au TP principal, inchangé)
2. **SplitIntervals** : découpe le génome en intervalles de taille équilibrée, pour paralléliser les étapes suivantes
3. **GenomicsDBImport**, exécuté sur chaque intervalle : importe l'ensemble des GVCF individuels dans un workspace GenomicsDB propre à cet intervalle
4. **GenotypeGVCFs**, exécuté sur chaque workspace GenomicsDB (un par intervalle) : produit un VCF multi-échantillon par intervalle
5. **GatherVcfs** : concatène les VCF obtenus sur chaque intervalle pour produire un unique VCF multi-échantillon sur le génome entier


## III - Étape par étape

In [ ]:
module load gatk4/4.6.1.0
mkdir -p Results/Variant_calling/large_cohort


### 1. HaplotypeCaller en mode GVCF (rappel)

Cette étape est strictement identique à celle du TP principal (partie III.3.2.a) : un GVCF par échantillon. Seule différence ici, puisqu'on se place dans le cas d'un "grand génome" : on ne restreint plus l'analyse à un intervalle réduit avec `--intervals`, on tourne sur l'ensemble du génome de référence.


In [ ]:
mkdir -p Results/Variant_calling/large_cohort/gvcf

for sample in milca_40995 milca_40998 control_41011 control_41012; do
  gatk HaplotypeCaller --java-options '-Xmx8G' \
     --input Results/Variant_calling/BaseRecalibration/${sample}_recal.bam \
     --reference data/genome/Bos_taurus_GCF_002263795.3_ARS-UCD2.0_chr2_chr6.fasta \
     --base-quality-score-threshold 18 \
     --min-base-quality-score 10 \
     --minimum-mapping-quality 20 \
     --sample-ploidy 2 \
     --emit-ref-confidence GVCF \
     --native-pair-hmm-threads 4 \
     --output Results/Variant_calling/large_cohort/gvcf/${sample}.g.vcf
done


### 2. SplitIntervals : découper le génome en intervalles équilibrés

[SplitIntervals](https://gatk.broadinstitute.org/hc/en-us/articles/30332044447003-SplitIntervals) permet de répartir le génome (ou une liste d'intervalles) en `N` sous-intervalles de taille comparable (en nombre de bases), pour paralléliser les étapes suivantes.

Options principales :
* `--scatter-count` : nombre de fichiers d'intervalles à produire (à adapter au nombre de cœurs/jobs disponibles, cf partie IV)
* `--subdivision-mode` : `BALANCING_WITHOUT_INTERVAL_SUBDIVISION` conserve les intervalles d'origine (utile si vous avez déjà un fichier de chromosomes) sans les recouper, contrairement au mode par défaut `INTERVAL_SUBDIVISION` qui peut couper un chromosome en plusieurs morceaux
* `--output` : répertoire de sortie (un fichier `.interval_list` par intervalle produit)


In [ ]:
mkdir -p Results/Variant_calling/large_cohort/intervals

gatk SplitIntervals \
   --reference data/genome/Bos_taurus_GCF_002263795.3_ARS-UCD2.0_chr2_chr6.fasta \
   --scatter-count 4 \
   --subdivision-mode BALANCING_WITHOUT_INTERVAL_SUBDIVISION \
   --output Results/Variant_calling/large_cohort/intervals


<div style="background-color: #e7f3fe; border-left: 6px solid #2196F3; padding: 10px;">
    ℹ️ <strong>Remarque</strong><br>
    Sur notre jeu réduit à 2 chromosomes (<code>chr2</code>/<code>chr6</code>), 4 intervalles suffisent pour illustrer le principe. Sur un génome bovin complet (29 autosomes + X), on utiliserait typiquement plusieurs dizaines à quelques centaines d'intervalles.
</div>


### 3. GenomicsDBImport : importer les GVCF, un workspace par intervalle

[GenomicsDBImport](https://gatk.broadinstitute.org/hc/en-us/articles/30332006200603-GenomicsDBImport) importe les GVCF de **tous** les échantillons dans une base GenomicsDB, propre à un intervalle donné.

Plutôt que de répéter `--variant` pour chaque échantillon, on utilise un fichier `--sample-name-map` : un fichier texte tabulé `nom_échantillon <TAB> chemin_du_gvcf`, une ligne par échantillon.

Options principales :
* `--genomicsdb-workspace-path` : chemin du workspace à créer (doit être un répertoire **inexistant**)
* `--sample-name-map` : fichier de correspondance échantillon → GVCF
* `--intervals` : l'intervalle à traiter (un des fichiers produits par `SplitIntervals`)
* `--batch-size` : nombre d'échantillons ouverts simultanément (0 = tous en même temps ; à préciser puis à réduire si vous manquez de mémoire sur de grandes cohortes)

<div style="background-color: #fff3cd; border-left: 6px solid #ffc107; color: #856404; padding: 10px; border-radius: 4px;">
    ⚠️Warning<br>
    L'espace disque temporaire utilisé par <code>GenomicsDBImport</code> peut dépasser l'espace disponible dans <code>/tmp</code> par défaut, en particulier avec de nombreux intervalles ou échantillons : pensez à préciser <code>--tmp-dir</code> vers un répertoire avec suffisamment d'espace.
</div>


In [ ]:
# Fichier de correspondance échantillon -> GVCF
for gvcf in Results/Variant_calling/large_cohort/gvcf/*.g.vcf; do
   sample=$(basename ${gvcf} | sed -E 's/\.g\.vcf$//')
   echo -e "${sample}\t${gvcf}" >> Results/Variant_calling/large_cohort/sample_map.tsv
done


In [ ]:
mkdir -p Results/Variant_calling/large_cohort/genomicsdb Results/Variant_calling/large_cohort/tmp

for interval in Results/Variant_calling/large_cohort/intervals/*-scattered.interval_list; do
   name=$(basename ${interval} .interval_list)
   gatk GenomicsDBImport --java-options '-Xmx8G' \
      --sample-name-map Results/Variant_calling/large_cohort/sample_map.tsv \
      --genomicsdb-workspace-path Results/Variant_calling/large_cohort/genomicsdb/${name} \
      --intervals ${interval} \
      --batch-size 50 \
      --reader-threads 4 \
      --tmp-dir Results/Variant_calling/large_cohort/tmp
done


### 4. GenotypeGVCFs : génotyper chaque workspace GenomicsDB

[GenotypeGVCFs](https://gatk.broadinstitute.org/hc/en-us/articles/30332044749979-GenotypeGVCFs) (utilisé dans le TP principal, cf partie 3.2.c) s'exécute ensuite une fois par intervalle, en pointant directement sur le workspace GenomicsDB correspondant, via l'URI `gendb://`.


In [ ]:
mkdir -p Results/Variant_calling/large_cohort/vcf_by_interval

for interval in Results/Variant_calling/large_cohort/intervals/*-scattered.interval_list; do
   name=$(basename ${interval} .interval_list)
   gatk GenotypeGVCFs --java-options '-Xmx8G' \
      --reference data/genome/Bos_taurus_GCF_002263795.3_ARS-UCD2.0_chr2_chr6.fasta \
      --variant gendb://Results/Variant_calling/large_cohort/genomicsdb/${name} \
      -stand-call-conf 30 \
      --tmp-dir Results/Variant_calling/large_cohort/tmp \
      --output Results/Variant_calling/large_cohort/vcf_by_interval/${name}.vcf.gz
done


### 5. GatherVcfs : reconstituer le VCF génome entier

[GatherVcfs](https://gatk.broadinstitute.org/hc/en-us/articles/30332005382555-GatherVcfs-Picard) concatène les VCF par intervalle en un seul VCF, à condition qu'ils soient fournis **dans l'ordre génomique** et qu'ils partagent tous le même ensemble d'échantillons.

<div style="background-color: #e7f3fe; border-left: 6px solid #2196F3; padding: 10px;">
    ℹ️ <strong>Remarque</strong><br>
    Les fichiers <code>0000-scattered.vcf.gz</code>, <code>0001-scattered.vcf.gz</code>, ... produits ci-dessus sont déjà dans l'ordre génomique (<code>SplitIntervals</code> les numérote dans l'ordre du <code>.dict</code> de la référence) : inutile de les trier.
</div>


In [ ]:
INPUTS=""
for vcf in Results/Variant_calling/large_cohort/vcf_by_interval/*.vcf.gz; do
   INPUTS="${INPUTS} --INPUT ${vcf}"
done

gatk GatherVcfs ${INPUTS} --java-options '-Xmx8G' \
   --OUTPUT Results/Variant_calling/large_cohort/4samples_genome.vcf.gz


## IV - Combien d'intervalles choisir (`--scatter-count`) ?

Il n'y a pas de valeur unique : le bon compromis dépend de plusieurs facteurs.

* **Trop peu d'intervalles** : peu de parallélisation, on ne profite pas des cœurs/jobs disponibles.
* **Trop d'intervalles** : chaque `GenomicsDBImport`/`GenotypeGVCFs` a un coût fixe de démarrage (ouverture des GVCF, écriture des métadonnées) ; multiplier les petits intervalles peut faire perdre plus de temps en overhead qu'on en gagne en parallélisation, et multiplie le nombre de petits fichiers à gérer.

<div style="background-color: #dafbe1; border-left: 6px solid #1f883d; padding: 10px;">
    💡 <strong>Astuce</strong><br>
    Un bon point de départ consiste à viser un nombre d'intervalles proche du nombre de jobs que vous pouvez soumettre en parallèle sur votre cluster (ou un petit multiple), tout en gardant chaque intervalle assez grand pour amortir le coût de démarrage (typiquement plusieurs dizaines de Mb par intervalle sur un génome de mammifère).
</div>


## V - Ajouter de nouveaux échantillons sans tout refaire

L'un des principaux intérêts de `GenomicsDBImport` par rapport à `CombineGVCFs` : un workspace GenomicsDB existant peut être mis à jour de façon incrémentale avec l'option `--genomicsdb-update-workspace-path` (au lieu de `--genomicsdb-workspace-path`), pour ajouter les GVCF de nouveaux échantillons sans réimporter les échantillons déjà présents.

<div style="background-color: #fff3cd; border-left: 6px solid #ffc107; color: #856404; padding: 10px; border-radius: 4px;">
    ⚠️Warning<br>
    Une mise à jour incrémentale qui échoue en cours de route peut laisser le workspace dans un état incohérent. Sauvegardez toujours votre workspace GenomicsDB avant d'y ajouter de nouveaux échantillons.
</div>


## VI - Références

* [GATK - SplitIntervals](https://gatk.broadinstitute.org/hc/en-us/articles/30332044447003-SplitIntervals)
* [GATK - GenomicsDBImport](https://gatk.broadinstitute.org/hc/en-us/articles/30332006200603-GenomicsDBImport)
* [GATK - GenotypeGVCFs](https://gatk.broadinstitute.org/hc/en-us/articles/30332044749979-GenotypeGVCFs)
* [GATK - GatherVcfs (Picard)](https://gatk.broadinstitute.org/hc/en-us/articles/30332005382555-GatherVcfs-Picard)
* [GATK - The logic of joint calling for germline short variants](https://gatk.broadinstitute.org/hc/en-us/articles/360035890431-The-logic-of-joint-calling-for-germline-short-variants)
* [GATK - HaplotypeCaller](https://gatk.broadinstitute.org/hc/en-us/articles/30332006386459-HaplotypeCaller)
* [GATK - CombineGVCFs (procédure en 3 étapes, TP principal)](https://gatk.broadinstitute.org/hc/en-us/articles/30332058987163-CombineGVCFs)
